# Sepsis Early Detection Model: A NEWS2-Based ML Approach
---
**Author:** Muhammed Omarjee (MBBS, King's College London 2023)

**Objective:** Build and evaluate machine learning models that predict sepsis onset from NEWS2 vital signs and routine laboratory markers, comparing multiple algorithms with clinically relevant evaluation metrics.

**Clinical context:** The UK National Early Warning Score (NEWS2) is a standardised tool for detecting acute deterioration. This project investigates whether ML models trained on NEWS2 parameters plus common labs (WCC, lactate, CRP) can flag sepsis risk earlier than the aggregate NEWS2 score alone.

> ⚠️ **Disclaimer:** This project uses synthetic data for educational and portfolio purposes. It is not a validated clinical decision-support tool.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
RANDOM_STATE = 42
print("All imports loaded successfully.")

## 1. Data Loading & Overview

The dataset contains 2,000 synthetic patients with a 20% sepsis prevalence, generated with physiologically plausible vital-sign distributions derived from published NEWS2 reference ranges (RCP, 2017) and Sepsis-3 criteria (Singer et al., JAMA 2016).

In [ ]:
df = pd.read_csv("data/raw/sepsis_cohort.csv")
print(f"Dataset shape: {df.shape}")
print(f"Sepsis prevalence: {df['sepsis_onset'].mean():.1%}\n")
df.head(10)

In [ ]:
df.describe().round(2)

In [ ]:
df.info()
print(f"\nMissing values:\n{df.isnull().sum()}")

## 2. Exploratory Data Analysis

### 2.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['sepsis_onset'].value_counts()
colors = ['#5DCAA5', '#E24B4A']
axes[0].bar(['No sepsis (0)', 'Sepsis (1)'], counts.values, color=colors, edgecolor='white')
axes[0].set_title('Class distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

for label, colour, name in [(0, '#5DCAA5', 'No sepsis'), (1, '#E24B4A', 'Sepsis')]:
    subset = df[df['sepsis_onset'] == label]['news2_score']
    axes[1].hist(subset, bins=20, alpha=0.65, color=colour, label=name, edgecolor='white')
axes[1].set_title('NEWS2 score by outcome')
axes[1].set_xlabel('NEWS2 score')
axes[1].legend()

plt.tight_layout()
plt.savefig('assets/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Vital Signs Distributions by Outcome

In [ ]:
vitals = ['resp_rate', 'spo2', 'heart_rate', 'systolic_bp', 'temperature', 'lactate']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, col in zip(axes.flatten(), vitals):
    for label, colour, name in [(0, '#5DCAA5', 'No sepsis'), (1, '#E24B4A', 'Sepsis')]:
        subset = df[df['sepsis_onset'] == label][col]
        ax.hist(subset, bins=25, alpha=0.6, color=colour, label=name, edgecolor='white')
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=8)

plt.suptitle('Vital signs & labs — sepsis vs non-sepsis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('assets/vitals_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Correlation Matrix

In [ ]:
numeric_cols = ['age', 'resp_rate', 'spo2', 'heart_rate', 'systolic_bp',
                'temperature', 'news2_score', 'wcc', 'lactate', 'crp', 'sepsis_onset']

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.savefig('assets/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Key Clinical Observations

The synthetic data exhibits expected clinical patterns:
- Sepsis patients show higher respiratory rates, heart rates, and temperatures
- SpO2 and systolic BP are lower in the sepsis group
- Lactate, WCC, and CRP are elevated — consistent with an inflammatory / organ dysfunction picture
- NEWS2 scores separate the groups well, but there is meaningful overlap in the 4–8 range

## 3. Preprocessing & Feature Engineering

In [ ]:
le_avpu = LabelEncoder()
df['avpu_encoded'] = le_avpu.fit_transform(df['avpu'])

le_sex = LabelEncoder()
df['sex_encoded'] = le_sex.fit_transform(df['sex'])

feature_cols = [
    'age', 'sex_encoded', 'resp_rate', 'spo2', 'heart_rate',
    'systolic_bp', 'temperature', 'avpu_encoded',
    'on_supplemental_o2', 'news2_score', 'wcc', 'lactate', 'crp'
]

X = df[feature_cols].copy()
y = df['sepsis_onset'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

print(f"Training set: {X_train.shape[0]} samples ({y_train.mean():.1%} sepsis)")
print(f"Test set:     {X_test.shape[0]} samples ({y_test.mean():.1%} sepsis)")

## 4. Model Training & Comparison

We compare three algorithms of increasing complexity:

| Model | Rationale |
|-------|-----------|
| **Logistic Regression** | Interpretable baseline; standard in clinical risk scores |
| **Random Forest** | Handles non-linear interactions; robust to outliers |
| **Gradient Boosting** | State-of-the-art tabular performance; captures complex patterns |

All models use stratified 5-fold cross-validation on the training set, with final evaluation on the held-out test set.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE, C=1.0
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=10,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        min_samples_leaf=10, random_state=RANDOM_STATE
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("5-fold cross-validation AUROC on training set:")
print("-" * 50)
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    print(f"{name:25s}  {scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
results = {}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    fitted_models[name] = model
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    results[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auroc': roc_auc_score(y_test, y_prob),
        'auprc': average_precision_score(y_test, y_prob),
        'brier': brier_score_loss(y_test, y_prob),
    }

summary = pd.DataFrame({
    name: {'AUROC': r['auroc'], 'AUPRC': r['auprc'], 'Brier Score': r['brier']}
    for name, r in results.items()
}).round(4).T

print("\nTest Set Performance Summary")
print("=" * 55)
print(summary.to_string())

## 5. Evaluation Plots

### 5.1 ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colours = ['#534AB7', '#1D9E75', '#D85A30']

for (name, r), c in zip(results.items(), colours):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={r['auroc']:.3f})", color=c, linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(loc='lower right', fontsize=9)

for (name, r), c in zip(results.items(), colours):
    prec, rec, _ = precision_recall_curve(y_test, r['y_prob'])
    axes[1].plot(rec, prec, label=f"{name} (AP={r['auprc']:.3f})", color=c, linewidth=2)
axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.3, label=f'Baseline ({y_test.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('assets/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Sepsis', 'Sepsis'],
                yticklabels=['No Sepsis', 'Sepsis'])
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion matrices (test set)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('assets/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Calibration Plot

A well-calibrated model means: when it predicts 70% probability of sepsis, roughly 70% of those patients actually have sepsis. This matters clinically because poorly calibrated probabilities mislead decision-making.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for (name, r), c in zip(results.items(), colours):
    prob_true, prob_pred = calibration_curve(y_test, r['y_prob'], n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, marker='o', label=f"{name} (Brier={r['brier']:.3f})", color=c, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfectly calibrated')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration curves')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('assets/calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Detailed Classification Reports

In [ ]:
for name, r in results.items():
    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, r['y_pred'], target_names=['No Sepsis', 'Sepsis']))

## 6. Feature Importance

Understanding which features drive predictions is critical for clinical trust. We use two approaches:
1. **Model-native importance** (Gini / coefficients)
2. **Permutation importance** (model-agnostic — measures AUROC drop when each feature is shuffled)

In [ ]:
best_name = max(results, key=lambda k: results[k]['auroc'])
best_model = fitted_models[best_name]
print(f"Best model by AUROC: {best_name} ({results[best_name]['auroc']:.4f})\n")

if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importance = np.abs(best_model.coef_[0])
else:
    importance = np.zeros(len(feature_cols))

imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importance}).sort_values('importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(imp_df['feature'], imp_df['importance'], color='#534AB7', edgecolor='white')
axes[0].set_title(f'{best_name} — native feature importance')
axes[0].set_xlabel('Importance')

perm_result = permutation_importance(
    best_model, X_test_scaled, y_test,
    n_repeats=20, random_state=RANDOM_STATE, scoring='roc_auc'
)
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std,
}).sort_values('importance_mean', ascending=True)

axes[1].barh(perm_df['feature'], perm_df['importance_mean'],
             xerr=perm_df['importance_std'], color='#1D9E75', edgecolor='white', capsize=3)
axes[1].set_title('Permutation importance (AUROC drop)')
axes[1].set_xlabel('Mean AUROC decrease')

plt.tight_layout()
plt.savefig('assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. NEWS2 Score as Baseline Classifier

To contextualise our ML models, we compare against the raw NEWS2 score as a standalone predictor. The RCP uses NEWS2 >= 5 for urgent clinical review, and >= 7 for emergency response.

In [ ]:
news2_auroc = roc_auc_score(y_test, X_test['news2_score'])
print(f"NEWS2 score alone — AUROC: {news2_auroc:.4f}")
print(f"Best ML model     — AUROC: {results[best_name]['auroc']:.4f}")
print(f"Improvement:        +{results[best_name]['auroc'] - news2_auroc:.4f}\n")

thresholds = range(0, 15)
fig, ax = plt.subplots(figsize=(8, 4))

sensitivities, specificities = [], []
for t in thresholds:
    pred = (X_test['news2_score'] >= t).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    tn = ((pred == 0) & (y_test == 0)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivities.append(sens)
    specificities.append(spec)

ax.plot(list(thresholds), sensitivities, 'o-', label='Sensitivity', color='#E24B4A', linewidth=2)
ax.plot(list(thresholds), specificities, 's-', label='Specificity', color='#534AB7', linewidth=2)
ax.axvline(x=5, color='grey', linestyle='--', alpha=0.5, label='NEWS2 >= 5 (urgent)')
ax.axvline(x=7, color='grey', linestyle=':', alpha=0.5, label='NEWS2 >= 7 (emergency)')
ax.set_xlabel('NEWS2 threshold')
ax.set_ylabel('Rate')
ax.set_title('NEWS2 threshold analysis — sensitivity vs specificity')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('assets/news2_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Best Model

In [ ]:
model_path = 'models/best_model.joblib'
joblib.dump({
    'model': best_model,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'label_encoders': {'avpu': le_avpu, 'sex': le_sex},
    'metadata': {
        'model_name': best_name,
        'auroc': results[best_name]['auroc'],
        'auprc': results[best_name]['auprc'],
        'train_size': len(X_train),
        'test_size': len(X_test),
    }
}, model_path)
print(f"Saved best model ({best_name}) -> {model_path}")

## 9. Limitations & Next Steps

### Limitations
- **Synthetic data:** Vital-sign distributions are modelled on published ranges but may not capture the full complexity of real clinical data (comorbidity interactions, medication effects, temporal trajectories).
- **Single time-point:** Real sepsis detection benefits from *trends* in vitals over time — this model uses a single snapshot.
- **No external validation:** Performance on this synthetic cohort does not generalise to real-world datasets without further validation.

### Next steps
1. **Real-world data:** Validate on MIMIC-IV or eICU Collaborative Research Database (both publicly accessible with appropriate credentials).
2. **Temporal modelling:** Use sequential vital signs with LSTM or Temporal Fusion Transformer to capture deterioration trajectories.
3. **Calibration tuning:** Apply Platt scaling or isotonic regression to improve probability calibration.
4. **Deployment prototype:** Build a lightweight API (FastAPI) that accepts NEWS2 parameters and returns a sepsis risk score.
5. **Fairness audit:** Evaluate model performance across demographic subgroups (age, sex) to identify potential disparities.